In [ ]:
from google.colab import files


import pandas as pd
import unicodedata
import re
import os
from datasets import Dataset


# 1: old dataset filename, replaced below
# if os.path.exists("Expanded_Intent_Dataset_2.csv"):
#   os.remove("Expanded_Intent_Dataset_2.csv")
if os.path.exists("Expanded_Intent_Dataset_3.csv"):  # 1
  os.remove("Expanded_Intent_Dataset_3.csv")  # 1




uploaded = files.upload()


# ✅ 1. Φόρτωση CSV ΜΙΑ φορά
# 1: old dataset filename, replaced below
# df_raw = pd.read_csv("Expanded_Intent_Dataset_2.csv", quotechar='"')
df_raw = pd.read_csv("Expanded_Intent_Dataset_3.csv", quotechar='"')  # 1

# ✅ 2. Normalization
def normalize_text(text):
    text = unicodedata.normalize('NFD', str(text))  # ensure string
    text = ''.join([c for c in text if unicodedata.category(c) != 'Mn'])  # remove accents
    return text.lower()

df = df_raw.copy()  # Κράτησε το raw ανέγγιχτο

# ✅ 3. Καθάρισμα text & intent
df["text"] = df["text"].apply(normalize_text)

df["intent"] = (
    df["intent"]
    .apply(normalize_text)
    .astype(str)
    .str.strip()
    .apply(lambda x: re.sub(r"[\t\n\r\s]+", "", x))       # remove whitespaces, tabs, newlines
    .apply(lambda x: re.sub(r"[^\w_α-ωΑ-Ω]", "", x))       # remove special characters except _
)

# ✅ 4. Έλεγχος μοναδικών intents και συχνοτήτων
print("📌 Μοναδικά intents:", df["intent"].nunique())
pd.set_option('display.max_rows', None)


print(df["intent"].value_counts())

# ✅ 5. Convert to HuggingFace Dataset
dataset = Dataset.from_pandas(df)

In [ ]:
!pip install -q datasets

from datasets import Dataset
from sklearn.preprocessing import LabelEncoder

import re

# Συνάρτηση για normalize
def normalize_text(text):
    text = unicodedata.normalize('NFD', text)
    text = ''.join([c for c in text if unicodedata.category(c) != 'Mn'])
    return text.lower()

df["text"] = df["text"].apply(normalize_text)

#df["intent"] = df["intent"].apply(normalize_text)

# Καθάρισε whitespaces και "σκουπίδια"
#df["intent"] = df["intent"].astype(str).str.strip().str.lower()

# Αφαίρεσε παράξενους χαρακτήρες
#df["intent"] = df["intent"].apply(lambda x: re.sub(r"[^a-zA-Z0-9_α-ωΑ-Ω]", "", x))

# Δες μοναδικές τιμές
print(df["intent"].unique())

# Κωδικοποίηση των intents σε αριθμούς
label_encoder = LabelEncoder()
df["label"] = label_encoder.fit_transform(df["intent"])

# ✅ Μετατροπή σε HuggingFace Dataset
dataset = Dataset.from_pandas(df)

# Έλεγχος
dataset

In [ ]:
# Χωρίζουμε σε train και test
dataset = dataset.train_test_split(test_size=0.2)

# Έλεγχος
dataset["train"][0]


In [ ]:
!pip install -q transformers

from transformers import AutoTokenizer

# Χρησιμοποιούμε multilingual μοντέλο γιατί έχουμε ελληνικά
# 2: old base model, replaced below
# model_name = "nlpaueb/bert-base-greek-uncased-v1"
model_name = "distilbert-base-multilingual-cased"  # 2
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Tokenization function
def tokenize(example):
    return tokenizer(example["text"], truncation=True)

# Εφαρμογή tokenization
tokenized_dataset = dataset.map(tokenize)

In [ ]:
from transformers import AutoModelForSequenceClassification

# Πόσα labels έχουμε;
num_labels = len(label_encoder.classes_)

# Φορτώνουμε το DistilBERT με σωστό αριθμό labels
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_labels
)


In [ ]:
!pip install -q evaluate

import evaluate
import numpy as np

accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1")

def compute_metrics(pred):
    logits, labels = pred
    predictions = np.argmax(logits, axis=-1)

    acc = accuracy.compute(predictions=predictions, references=labels)
    f1_macro = f1.compute(predictions=predictions, references=labels, average='macro')

    return {
        "accuracy": acc["accuracy"],
        "f1_macro": f1_macro["f1"]
    }


In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorWithPadding

training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch", # Changed save_strategy to match eval_strategy
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=7,
    weight_decay=0.01,
    load_best_model_at_end=True,
    logging_steps=10,


)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    processing_class=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=compute_metrics,
)

In [ ]:
import os
os.environ["WANDB_DISABLED"] = "true"

trainer.train()



In [ ]:
# αφού έχεις fit τον LabelEncoder
id2label = {i: lbl for i, lbl in enumerate(label_encoder.classes_)}
label2id = {lbl: i for i, lbl in id2label.items()}

model.config.id2label = id2label
model.config.label2id = label2id

# 3: old save directory, replaced below
# model.save_pretrained("dimos-intent-model")
# tokenizer.save_pretrained("dimos-intent-model")
model.save_pretrained("dimos-intent-model-distilbert")  # 3
tokenizer.save_pretrained("dimos-intent-model-distilbert")  # 3

# 3b: label_encoder.joblib is required by connector.py next to the model
# checkpoint; no cell previously saved it, so it's added here.
import joblib  # 3b
joblib.dump(label_encoder, "dimos-intent-model-distilbert/label_encoder.joblib")  # 3b

In [ ]:
from transformers import pipeline
import unicodedata

def normalize_text(s):
    s = unicodedata.normalize('NFD', str(s))
    s = ''.join(c for c in s if unicodedata.category(c) != 'Mn')
    return s.lower()

# 4: old model dir, replaced below
# clf = pipeline("text-classification", model="dimos-intent-model", tokenizer="dimos-intent-model")
clf = pipeline("text-classification", model="dimos-intent-model-distilbert", tokenizer="dimos-intent-model-distilbert")  # 4

text = "δεν εχω που να πεταξω τα σκουπιδια μου"
out = clf(normalize_text(text))[0]

predicted_label = out["label"]
score = out["score"]

print(out)
print(f'🗣 "{text}"\n👉 Predicted Intent: {predicted_label} (score: {score:.3f})')

In [ ]:
import pandas as pd
import unicodedata
import re
from sklearn.preprocessing import LabelEncoder

# ✅ Φόρτωση αρχείου
# 5: old dataset filename, replaced below
# df = pd.read_csv("Expanded_Intent_Dataset_2.csv", quotechar='"')
df = pd.read_csv("Expanded_Intent_Dataset_3.csv", quotechar='"')  # 5

# ✅ Normalization function
def normalize_text(text):
    text = unicodedata.normalize('NFD', str(text))
    text = ''.join([c for c in text if unicodedata.category(c) != 'Mn'])  # remove accents
    return text.lower()

# ✅ Καθαρισμός text & intent
df["text"] = df["text"].apply(normalize_text)

df["intent"] = (
    df["intent"]
    .apply(normalize_text)
    .astype(str)
    .str.strip()
    .apply(lambda x: re.sub(r"[\t\n\r\s]+", "", x))       # remove whitespaces, tabs, newlines
    .apply(lambda x: re.sub(r"[^\w_α-ωΑ-Ω]", "", x))       # remove special characters except _
)

# ✅ Κωδικοποίηση intents σε αριθμούς
label_encoder = LabelEncoder()
df["label"] = label_encoder.fit_transform(df["intent"])

# ✅ Εκτύπωση mapping label -> intent
id2intent = dict(enumerate(label_encoder.classes_))

print("🔢 Mapping από label σε intent:\n")
for label, intent in id2intent.items():
    print(f"{label:2} → {intent}")

In [ ]:
# 6: old model dir, replaced below
# !zip -r dimos-intent-model.zip dimos-intent-model/
!zip -r dimos-intent-model-distilbert.zip dimos-intent-model-distilbert/  # 6

In [ ]:
from transformers import pipeline
import unicodedata

def normalize_text(s):
    s = unicodedata.normalize('NFD', str(s))
    s = ''.join(c for c in s if unicodedata.category(c) != 'Mn')
    return s.lower()

# 7: old model dir, replaced below
# clf = pipeline("text-classification",
#                model="dimos-intent-model",
#                tokenizer="dimos-intent-model")
clf = pipeline("text-classification",
               model="dimos-intent-model-distilbert",  # 7
               tokenizer="dimos-intent-model-distilbert")  # 7

text = "η γυναικα μου γεννησε τι πρεπει να κανω τωρα"

# ΜΟΝΟ Top-1
top1 = clf(normalize_text(text), top_k=1)[0]
print(f'Top-1 → {top1["label"]} (score: {top1["score"]:.3f})')

# ή Top-5
top5 = clf(normalize_text(text), top_k=5)
for r in top5:
    print(f'{r["label"]}: {r["score"]:.3f}')